In [ ]:
# 1. Get the project code and install requirements (safe to re-run)
import importlib, importlib.metadata as md, os, subprocess, sys, time

REPO_URL = "https://github.com/Yash13606/sanjeevani.git"
REPO_DIR = "/content/sanjeevani"


def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True)


if not os.path.isdir(os.path.join(REPO_DIR, "ml")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True, capture_output=True)
    print("Cloned the project from GitHub.")
else:
    # an older session may hold an old copy of the project: bring it up to date with GitHub main
    git("fetch", "--depth", "1", "origin", "main")
    git("reset", "--hard", "origin/main")
    print("Project already present: updated it to the latest GitHub main.")
print("Project version (git commit):", git("rev-parse", "--short", "HEAD").stdout.strip())

os.chdir(os.path.join(REPO_DIR, "ml"))
print("Working directory:", os.getcwd())

# Install the exact package versions this project was tested with (requirements.txt).
# Colab ships its own pandas; pip may print "google-colab requires pandas==2.2.3". That line only concerns Colab's
# own built-in package. It is a warning, not an error in this project, so it is not shown here.
print("\nInstalling the project's packages (about a minute the first time, seconds afterwards)...")
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:]); print(result.stderr[-3000:])
    raise RuntimeError("pip could not install requirements.txt - see the messages above")
importlib.invalidate_caches()
print("Installed: " + " | ".join(f"{d} {md.version(d)}" for d in
                                 ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "xgboost"]))

# pip may have just replaced numpy / pandas on disk while the OLD versions are still loaded in this running session,
# which causes confusing errors later. If anything loaded is out of date, restart the runtime ONCE.
stale = []
for dist, module in [("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "sklearn"), ("scipy", "scipy"),
                     ("matplotlib", "matplotlib"), ("xgboost", "xgboost")]:
    loaded = sys.modules.get(module)
    if loaded is not None and getattr(loaded, "__version__", None) != md.version(dist):
        stale.append(f"{dist}: loaded {loaded.__version__}, installed {md.version(dist)}")
if stale:
    print("\nThe packages were updated to the versions this project needs:\n  " + "\n  ".join(stale))
    print("\n>>> Colab is restarting the runtime now. A 'session crashed' message is EXPECTED here. <<<")
    print(">>> When it is back, click Runtime > Run all ONE more time. It will then go straight through. <<<")
    sys.stdout.flush(); sys.stderr.flush(); time.sleep(3)      # let the message reach the screen first
    os.kill(os.getpid(), 9)
else:
    print("\nAll packages are consistent. Ready - continue with the next cell.")

In [ ]:
# 2. Run the tests, then simulate the synthetic worlds and train/evaluate the official M1/M2 benchmark
import datetime, os, subprocess, sys

RUN_TESTS = True                  # set False to skip the pytest check
# Change ONE word to switch the study:
#   "smoke" = 1 train / 1 validation / 1 test world - use it first to check everything runs.
#             This cell takes roughly 15 minutes; the whole notebook roughly an hour on Colab's free 2-CPU machine.
#   "full"  = 20 train / 5 validation / 10 test worlds (several hours: keep this browser tab open)
STUDY = "smoke"
CONFIG = {"smoke": "config/smoke.yaml", "full": "config/simulation_config.yaml"}[STUDY]
print(f"Study: {STUDY} ({CONFIG})")


def run_streaming(cmd):
    """Run a command, show its output live, and raise if it fails."""
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", env=env)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")


RUN_DIR = os.path.abspath(
    "runs/colab-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)

if RUN_TESTS:
    run_streaming([sys.executable, "-m", "pytest", "-q", "tests"])
run_streaming([sys.executable, "-u", "-m", "sanjeevani_ml.train_and_evaluate",
               "--config", CONFIG, "--output-dir", RUN_DIR])
run_streaming([sys.executable, "write_manifest.py", "--config", CONFIG, "--run-root", RUN_DIR])
print("\nSimulation complete! Results saved in:", RUN_DIR)

In [ ]:
!pip install -q seaborn matplotlib

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

# ==============================================================================
# REVIEW II: LEAK-FREE ML BENCHMARK (train worlds -> validation worlds -> test worlds)
# ==============================================================================
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             confusion_matrix, roc_curve, average_precision_score)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# ==============================================================================
# 1. MULTIPLE LINEAR REGRESSION (Outbreak Severity Index, scaled 0-100)
# ==============================================================================
print("=" * 55)
print("1. MULTIPLE LINEAR REGRESSION EVALUATION")
print("=" * 55)
# Environmental / infrastructure predictors only: the case-count windows are excluded
# because the severity target itself is built from reported cases.
reg_features = [c for c in cls_cols if not c.startswith(("asha_", "opd_", "idsp_"))]

reg_imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, reg_features])
X_reg_train = pd.DataFrame(reg_imputer.transform(df.loc[train_mask, reg_features]), columns=reg_features)
X_reg_test  = pd.DataFrame(reg_imputer.transform(df.loc[test_mask,  reg_features]), columns=reg_features)

raw_train = df.loc[train_mask, "asha_cases_7d"].fillna(0).to_numpy()
raw_test  = df.loc[test_mask,  "asha_cases_7d"].fillna(0).to_numpy()
max_ref   = max(raw_train.max(), 1.0)                  # scale with a TRAIN-only reference
y_sev_train = raw_train / max_ref * 100
y_sev_test  = raw_test  / max_ref * 100

reg_model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_reg_train, y_sev_train)
y_pred_sev = np.clip(reg_model.predict(X_reg_test), 0, 100)

mse  = mean_squared_error(y_sev_test, y_pred_sev)
rmse = float(np.sqrt(mse))
mae  = mean_absolute_error(y_sev_test, y_pred_sev)
r2   = r2_score(y_sev_test, y_pred_sev)

reg_df = pd.DataFrame({
    "Regression Metric": ["MSE (Severity Index 0-100)", "RMSE (Severity Index)", "MAE (Severity Index)",
                          "R2 Score", "Physical RMSE (Cases)"],
    "Value": [round(mse, 2), round(rmse, 2), round(mae, 2), round(r2, 4), f"{rmse / 100 * max_ref:.1f} cases"],
})
display(reg_df)
reg_df.to_csv("review2_regression_metrics.csv", index=False)

# ==============================================================================
# 2. CLASSIFIERS: fit on train worlds, pick the F1 threshold on val worlds, score test worlds
# ==============================================================================
print("\n" + "=" * 55)
print("2. COMPARISON OF CLASSIFICATION MODELS")
print("   (validation-chosen threshold -> test evaluation)")
print("=" * 55)
print("Note: K-fold CV below re-uses the training worlds (an optimistic ceiling).")
print("      The test worlds are independent simulations (true generalisation).\n")

pos_ratio = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
classifiers = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=SEED)),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, class_weight="balanced",
                                            random_state=SEED, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=120, max_depth=5, learning_rate=0.08,
                             scale_pos_weight=pos_ratio, eval_metric="logloss",
                             random_state=SEED, n_jobs=-1),
}

cls_results, cls_preds, cls_probs = [], {}, {}
for name, clf in classifiers.items():
    row, test_probs, preds = evaluate_classifier(name, clf, X_train, y_train, X_val, y_val, X_test, y_test)
    row["Val. Avg. Precision"] = round(average_precision_score(y_val, model_scores(clf, X_val)), 4)
    cls_results.append(row)
    cls_preds[name], cls_probs[name] = preds, test_probs

cls_df = pd.DataFrame(cls_results)[["Model", "Val. Avg. Precision", "Test Accuracy", "Test Precision",
                                    "Test Recall", "Test F1-Score", "Test ROC-AUC", "Operating Threshold"]]
display(cls_df)
cls_df.to_csv("review2_classification_results.csv", index=False)

best_model = cls_df.loc[cls_df["Val. Avg. Precision"].idxmax(), "Model"]
print(f"\nSelected model (by validation average precision): {best_model}")

# ==============================================================================
# 3. K-FOLD CROSS-VALIDATION ON THE TRAINING WORLDS (optimistic ceiling)
# ==============================================================================
print("\n" + "=" * 55)
print("3. K-FOLD CROSS-VALIDATION (5-FOLD ON TRAINING WORLDS, GROUPED BY WEEK)")
print("   This is the WITHIN-WORLD performance ceiling.")
print("=" * 55)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
kfold_rows, cv_f1 = [], {}
for name, clf in classifiers.items():
    cv = cross_validate(clf, X_train, y_train, cv=sgkf, groups=week_train,
                        scoring=["accuracy", "precision", "recall", "f1"])
    cv_f1[name] = float(cv["test_f1"].mean())
    kfold_rows.append({"Model": name, **{
        f"Mean {label}": f"{cv[f'test_{key}'].mean():.4f} (+/-{cv[f'test_{key}'].std():.4f})"
        for label, key in [("Accuracy", "accuracy"), ("Precision", "precision"),
                           ("Recall", "recall"), ("F1-Score", "f1")]}})
kfold_df = pd.DataFrame(kfold_rows)
display(kfold_df)
kfold_df.to_csv("review2_kfold_cv.csv", index=False)

# ==============================================================================
# 4. BOOTSTRAP (95% CI on the test worlds, resampling whole WEEKS)
#    Rows of the same week are correlated, so resampling single rows would give CIs that are too narrow.
# ==============================================================================
print("\n" + "=" * 55)
print("4. BOOTSTRAPPING & RESAMPLING (500 iterations on the test worlds)")
print("=" * 55)
codes, weeks = pd.factorize(week_test)
rows_of_week = [np.flatnonzero(codes == k) for k in range(len(weeks))]
rng = np.random.default_rng(SEED)
best_preds = cls_preds[best_model]
boot = {"acc": [], "prec": [], "rec": [], "f1": [], "mse": []}
for _ in range(500):
    pick = rng.integers(0, len(weeks), len(weeks))
    ic = np.concatenate([rows_of_week[k] for k in pick])
    boot["acc"].append(accuracy_score(y_test[ic], best_preds[ic]))
    boot["prec"].append(precision_score(y_test[ic], best_preds[ic], zero_division=0))
    boot["rec"].append(recall_score(y_test[ic], best_preds[ic], zero_division=0))
    boot["f1"].append(f1_score(y_test[ic], best_preds[ic], zero_division=0))
    boot["mse"].append(mean_squared_error(y_sev_test[ic], y_pred_sev[ic]))


def _boot_row(label, estimate, values, digits=4):
    return [label, round(estimate, digits), round(np.mean(values), digits),
            round(np.percentile(values, 2.5), digits), round(np.percentile(values, 97.5), digits)]


boot_df = pd.DataFrame([
    _boot_row("Accuracy",  accuracy_score(y_test, best_preds), boot["acc"]),
    _boot_row("Precision", precision_score(y_test, best_preds, zero_division=0), boot["prec"]),
    _boot_row("Recall",    recall_score(y_test, best_preds, zero_division=0), boot["rec"]),
    _boot_row("F1-Score",  f1_score(y_test, best_preds, zero_division=0), boot["f1"]),
    _boot_row("Reg. MSE",  mse, boot["mse"], 2),
], columns=["Metric", f"{best_model} Estimate", "Bootstrap Mean", "95% CI Lower", "95% CI Upper"])
display(boot_df)
boot_df.to_csv("review2_bootstrap_ci.csv", index=False)

# ==============================================================================
# 5. ALL 5 PLOTS
# ==============================================================================
# Plot 1: regression
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y_sev_test, y_pred_sev, alpha=0.2, color="#1f77b4", edgecolor="none", s=15)
axes[0].plot([0, 100], [0, 100], "r--", lw=2, label="Ideal y=x")
axes[0].set_title(f"Linear Regression: Outbreak Severity (R2={r2:.3f}, RMSE={rmse:.1f})")
axes[0].set_xlabel("Actual Severity Index (0-100)")
axes[0].set_ylabel("Predicted Severity Index (0-100)")
axes[0].set_xlim(-2, 102); axes[0].set_ylim(-2, 102)
axes[0].legend()
axes[1].hist(y_sev_test - y_pred_sev, bins=30, color="#2ca02c", edgecolor="k", alpha=0.7)
axes[1].axvline(0, color="r", linestyle="--", lw=2)
axes[1].set_title(f"Residual Distribution (MAE={mae:.1f})")
axes[1].set_xlabel("Residual (Actual - Predicted)")
plt.tight_layout(); plt.savefig("review2_regression_analysis.png", dpi=300); plt.show()

# Plot 2: confusion matrices
n_models = len(cls_preds)
fig, axes = plt.subplots(1, n_models, figsize=(4.5 * n_models, 4.2))
for ax, (name, preds) in zip(np.atleast_1d(axes), cls_preds.items()):
    cm = confusion_matrix(y_test, preds, labels=[0, 1])
    ax.matshow(cm, cmap="Blues", alpha=0.7)
    for (m, n), v in np.ndenumerate(cm):
        ax.text(n, m, f"{v}", ha="center", va="center", fontsize=11, weight="bold")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Outbreak", "Outbreak"]); ax.set_yticklabels(["No Outbreak", "Outbreak"])
    ax.set_title(f"{name}\nF1={cls_df.loc[cls_df['Model'] == name, 'Test F1-Score'].values[0]:.3f}", pad=12)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig("review2_confusion_matrices.png", dpi=300); plt.show()

# Plot 3: grouped bar comparison
fig, ax = plt.subplots(figsize=(10, 4.8))
metrics = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1-Score"]
x = np.arange(len(metrics)); width = 0.8 / n_models
for i, (_, row) in enumerate(cls_df.iterrows()):
    ax.bar(x + (i - (n_models - 1) / 2) * width, [row[m] for m in metrics], width, label=row["Model"])
ax.set_xticks(x); ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1-Score"])
ax.set_ylim(0, 1.15)
ax.set_title("Model Comparison on Independent Test Worlds", fontsize=12, weight="bold")
ax.legend(loc="upper right")
plt.tight_layout(); plt.savefig("review2_classification_comparison.png", dpi=300); plt.show()

# Plot 4: ROC & PR curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, probs in cls_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={safe_auc(y_test, probs):.2f})", lw=2)
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC Curves (Test Worlds)"); axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate"); axes[0].legend()
for name, probs in cls_probs.items():
    p, r, _ = precision_recall_curve(y_test, probs)
    axes[1].plot(r, p, label=name, lw=2)
axes[1].set_title("Precision-Recall Curves (Test Worlds)")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout(); plt.savefig("review2_roc_pr_curves.png", dpi=300); plt.show()

# Plot 5: bootstrap distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, key, label, colour in [(axes[0], "rec", "Recall", "#9467bd"), (axes[1], "f1", "F1-Score", "#e377c2")]:
    ax.hist(boot[key], bins=25, color=colour, edgecolor="k", alpha=0.7)
    ax.axvline(np.percentile(boot[key], 2.5), color="r", linestyle="--", label="95% CI lower")
    ax.axvline(np.percentile(boot[key], 97.5), color="r", linestyle="--", label="95% CI upper")
    ax.set_title(f"Bootstrap: {best_model} {label} distribution"); ax.set_xlabel(label); ax.legend()
plt.tight_layout(); plt.savefig("review2_bootstrap_distributions.png", dpi=300); plt.show()

# ==============================================================================
# 6. SUMMARY (every number below is computed from this run)
# ==============================================================================
row = cls_df.loc[cls_df["Model"] == best_model].iloc[0]
gap = cv_f1[best_model] - row["Test F1-Score"]
print("\nSUCCESS! All 5 plots generated.\n")
print("Summary for presentation:")
print(f"  Best model ({best_model}) on the held-out test worlds:")
print(f"    Accuracy = {row['Test Accuracy']:.1%}, Recall = {row['Test Recall']:.1%}")
print(f"    F1-Score = {row['Test F1-Score']:.1%}, ROC-AUC = {row['Test ROC-AUC']:.4f}")
print(f"  Within-world K-fold F1 = {cv_f1[best_model]:.1%} vs test-world F1 = {row['Test F1-Score']:.1%} "
      f"(gap {gap:+.1%}).")
if gap > 0.02:
    print("  The K-fold score is higher because it re-uses the training worlds; the test worlds are "
          "unseen simulations, so the test score is the honest estimate.")
else:
    print("  K-fold and test-world scores are close in this run.")

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, roc_curve,
                             average_precision_score)

# ==============================================================================
# REVIEW II (B): STRONGER MODELS - train on train+val worlds, test worlds stay held out
# The operating threshold is chosen from OUT-OF-FOLD predictions (never from the data the
# model was fitted on, which would give over-confident probabilities and a bad threshold).
# ==============================================================================
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_predict

dev_mask = train_mask | val_mask
dev_imputer = SimpleImputer(strategy="median").fit(df.loc[dev_mask, cls_cols])
X_dev  = pd.DataFrame(dev_imputer.transform(df.loc[dev_mask,  cls_cols]), columns=cls_cols)
X_test_dev = pd.DataFrame(dev_imputer.transform(df.loc[test_mask, cls_cols]), columns=cls_cols)
y_dev  = df.loc[dev_mask, LABEL].to_numpy()
week_dev = df.loc[dev_mask, "date"].to_numpy()

print(f"Development (train+val): {len(X_dev)} rows, Test: {len(X_test_dev)} rows")
print(f"Positive rate - Dev: {y_dev.mean():.2%}, Test: {y_test.mean():.2%}")

pos_ratio_dev = (len(y_dev) - y_dev.sum()) / max(y_dev.sum(), 1)
lr  = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
rf  = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=5,
                             class_weight="balanced", random_state=SEED, n_jobs=-1)
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.06, subsample=0.85, colsample_bytree=0.85,
                    scale_pos_weight=pos_ratio_dev, eval_metric="logloss", random_state=SEED, n_jobs=-1)
ensemble = VotingClassifier(estimators=[("lr", lr), ("rf", rf), ("xgb", xgb)], voting="soft")

sgkf_dev = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

print("\n5-fold CV on the development worlds (grouped by week):")
for name, clf in [("XGBoost", xgb), ("Random Forest", rf), ("Ensemble (LR+RF+XGB)", ensemble)]:
    cv = cross_validate(clf, X_dev, y_dev, cv=sgkf_dev, groups=week_dev,
                        scoring=["accuracy", "f1", "recall", "precision"])
    print(f"  {name}: Acc={cv['test_accuracy'].mean():.4f}(+/-{cv['test_accuracy'].std():.4f})  "
          f"F1={cv['test_f1'].mean():.4f}(+/-{cv['test_f1'].std():.4f})  "
          f"Recall={cv['test_recall'].mean():.4f}(+/-{cv['test_recall'].std():.4f})")

# out-of-fold probabilities -> threshold; then refit on the whole development set -> test
thresholds, final_probs = {}, {}
for name, clf in [("XGBoost (tuned)", xgb), ("Ensemble LR+RF+XGB", ensemble)]:
    oof = cross_val_predict(clf, X_dev, y_dev, cv=sgkf_dev, groups=week_dev, method="predict_proba")[:, 1]
    thresholds[name] = best_f1_threshold(y_dev, oof)
    clf.fit(X_dev, y_dev)
    final_probs[name] = clf.predict_proba(X_test_dev)[:, 1]

print("\n" + "=" * 55)
print("TEST WORLD RESULTS (stronger models)")
print("=" * 55)
results = []
for name, probs in final_probs.items():
    preds = (probs >= thresholds[name]).astype(int)
    results.append({
        "Model": name,
        "Test Accuracy":  round(accuracy_score(y_test, preds), 4),
        "Test Precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "Test Recall":    round(recall_score(y_test, preds, zero_division=0), 4),
        "Test F1-Score":  round(f1_score(y_test, preds, zero_division=0), 4),
        "Test ROC-AUC":   round(safe_auc(y_test, probs), 4),
        "Operating Threshold": round(thresholds[name], 3),
    })
final_df = pd.DataFrame(results)
display(final_df)
final_df.to_csv("review2_final_models.csv", index=False)

# ROC and PR comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, probs in final_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={safe_auc(y_test, probs):.3f})", lw=2)
    p, r, _ = precision_recall_curve(y_test, probs)
    axes[1].plot(r, p, label=name, lw=2)
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC: XGBoost vs Ensemble (Test Worlds)")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate"); axes[0].legend()
axes[1].set_title("Precision-Recall: XGBoost vs Ensemble (Test Worlds)")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout(); plt.savefig("review2_smoke_best.png", dpi=300); plt.show()

n_train_worlds = df.loc[train_mask, "world_id"].nunique()
print("\n" + "=" * 55)
print(f"This run used {n_train_worlds} training world(s), {df.loc[val_mask, 'world_id'].nunique()} validation "
      f"world(s) and {df.loc[test_mask, 'world_id'].nunique()} test world(s).")
print("For the full study set CONFIG = 'config/simulation_config.yaml' in cell 2 and re-run:")
print("20 training / 5 validation / 10 test worlds. Compare its numbers with this run, don't assume them.")
print("=" * 55)

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

# ==============================================================================
# OFFICIAL PROJECT BENCHMARK SUMMARY (M1 early warning & M2 detection)
# Produced by sanjeevani_ml.train_and_evaluate in cell 2 (all models, all surveillance scenarios)
# ==============================================================================
csv_path = os.path.join(latest_run, "results", "benchmark_metrics_summary.csv")
if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"{csv_path} not found - run cell 2 first.")
df_bm = pd.read_csv(csv_path)

print("=" * 75)
print("OFFICIAL SANJEEVANI BENCHMARK RESULTS (M1: Early Warning vs M2: Detection)")
print("=" * 75)
display_cols = [c for c in ["Task", "Model", "Scenario", "Rows", "Positives", "AveragePrecision", "ROCAUC",
                            "F1AtValThreshold", "PrecisionAtValThreshold", "RecallAtValThreshold"]
                if c in df_bm.columns]
selected_df = df_bm[df_bm["Selected"].astype(str).str.lower() == "true"][display_cols].reset_index(drop=True)
display(selected_df)
selected_df.to_csv("official_benchmark_selected.csv", index=False)
print("\nSaved summary as 'official_benchmark_selected.csv'")

# Model comparison on all test scenarios pooled, per task
pooled = df_bm[df_bm["Scenario"] == "all"][["Task", "Model", "Selected", "AveragePrecision", "ROCAUC",
                                            "F1AtValThreshold", "RecallAtValThreshold"]]
print("\nAll candidate models vs baselines (all scenarios pooled, test worlds):")
display(pooled.sort_values(["Task", "AveragePrecision"], ascending=[True, False]).reset_index(drop=True))
pooled.to_csv("official_benchmark_all_models.csv", index=False)

In [ ]:
# ==============================================================================
# ZIP & DOWNLOAD THE REVIEW II ARTIFACTS
# ==============================================================================
import glob, os, shutil

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:           # running somewhere other than Colab: just build the zip
    IN_COLAB = False

OUT_ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
pack_dir = os.path.join(OUT_ROOT, "sanjeevani_review2_presentation_pack")
shutil.rmtree(pack_dir, ignore_errors=True)
os.makedirs(pack_dir)

graphs = sorted(glob.glob("review2_*.png"))
tables = sorted(glob.glob("review2_*.csv") + glob.glob("official_benchmark_*.csv"))
for path in graphs + tables:
    shutil.copy(path, pack_dir)
print(f"Collected {len(graphs)} plots:")
for g in graphs:
    print("  -", g)
print(f"Collected {len(tables)} tables:")
for t in tables:
    print("  -", t)

zip_path = shutil.make_archive(os.path.join(OUT_ROOT, "Sanjeevani_Review2_Final_Pack"), "zip", pack_dir)
print("\nCreated:", zip_path)
if IN_COLAB:
    files.download(zip_path)
    print("Download started - check your computer's Downloads folder.")

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

# ==============================================================================
# DISPLAY DATASET FOR FACULTY REVIEW
# ==============================================================================
# ---- dataset preview (for faculty review) ----
preview_cols = [c for c in ["date", "unit_id", "rain_7d", "temp_mean_7d", "turbidity_mean_7d",
                            "chlorine_mean_7d", "ecoli_last_cfu", "asha_cases_7d",
                            "label_m1_onset_7d", "label_m2_active"] if c in df.columns]
sample_world = df[(df["split"] == "train")].iloc[:10]
print(f"\nDataset preview ({df['world_id'].nunique()} worlds, {len(df)} unit-week rows, "
      f"{df.shape[1]} columns):")
display(sample_world[preview_cols])

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

# ==============================================================================
# REVIEW III - PART 1: PCA, KNN, NAIVE BAYES (GAUSSIAN / MULTINOMIAL / BERNOULLI), SVM KERNELS
# Same protocol as Review II: fit on train worlds, F1 threshold from validation worlds, report on test worlds.
# ==============================================================================
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.svm import SVC

print(f"Data: Train = {len(X_train)} rows, Val = {len(X_val)} rows, Test = {len(X_test)} rows, "
      f"{len(cls_cols)} features (task {TASK.upper()}).")

# ------------------------------------------------------------------------------
# 1. PRINCIPAL COMPONENT ANALYSIS (fitted on the training worlds only)
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("1. PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("=" * 60)
pca_scaler = StandardScaler().fit(X_train)
Z_train, Z_test = pca_scaler.transform(X_train), pca_scaler.transform(X_test)

N_PC = 10
pca = PCA(n_components=N_PC, random_state=SEED).fit(Z_train)
Z_test_pca = pca.transform(Z_test)
cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
for i in range(5):
    print(f"  PC{i + 1}: {pca.explained_variance_ratio_[i] * 100:.2f}% variance (cumulative {cum_var[i]:.2f}%)")


def top_loadings(component, n=3):
    order = np.argsort(-np.abs(pca.components_[component]))[:n]
    return ", ".join(cls_cols[i] for i in order)


pc1_label, pc2_label = top_loadings(0), top_loadings(1)
print(f"  PC1 is driven mainly by: {pc1_label}")
print(f"  PC2 is driven mainly by: {pc2_label}")

# ------------------------------------------------------------------------------
# 2. KNN, NAIVE BAYES (3 variants) and SVM KERNELS
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("2. KNN, NAIVE BAYES & SVM KERNEL CLASSIFIERS")
print("=" * 60)

# SVMs are fitted on a fixed random subsample so that training finishes quickly in Colab
rng = np.random.default_rng(SEED)
sub_idx = rng.choice(len(X_train), size=min(6000, len(X_train)), replace=False)
X_sub, y_sub = X_train.iloc[sub_idx], y_train[sub_idx]

part1_models = {
    "K-Nearest Neighbours (k=5)": (make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)), False),
    "Gaussian Naive Bayes":       (make_pipeline(StandardScaler(), GaussianNB()), False),
    "Bernoulli Naive Bayes":      (make_pipeline(StandardScaler(), BernoulliNB(binarize=0.0)), False),
    # MinMax scaling gives the non-negative counts-like input MultinomialNB needs (clip guards unseen ranges)
    "Multinomial Naive Bayes":    (make_pipeline(MinMaxScaler(clip=True), MultinomialNB()), False),
    "SVM (Linear Kernel)":        (make_pipeline(StandardScaler(), SVC(kernel="linear", class_weight="balanced")), True),
    "SVM (RBF Kernel)":           (make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced")), True),
    "SVM (Polynomial Kernel d=3)": (make_pipeline(StandardScaler(), SVC(kernel="poly", degree=3, class_weight="balanced")), True),
}

part1_rows, part1_scores = [], {}
for name, (model, use_subsample) in part1_models.items():
    print(f"  Fitting {name}...")
    X_fit, y_fit = (X_sub, y_sub) if use_subsample else (X_train, y_train)
    row, s_te, _ = evaluate_classifier(name, model, X_fit, y_fit, X_val, y_val, X_test, y_test)
    part1_rows.append(row)
    part1_scores[name] = s_te

part1_df = pd.DataFrame(part1_rows)
display(part1_df)
part1_df.to_csv("review3_part1_classifiers.csv", index=False)

# ------------------------------------------------------------------------------
# 3. PLOTS: SCREE PLOT AND 2-D PCA PROJECTION
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, N_PC + 1), pca.explained_variance_ratio_ * 100, color="#1f77b4", alpha=0.7,
            label="Individual variance")
axes[0].step(range(1, N_PC + 1), cum_var, where="mid", color="red", lw=2, label="Cumulative variance")
axes[0].set_title("PCA Scree Plot: Explained Variance", fontsize=11, weight="bold")
axes[0].set_xlabel("Principal Component Index"); axes[0].set_ylabel("Explained Variance (%)")
axes[0].legend(loc="center right")

pts = rng.choice(len(Z_test_pca), size=min(1200, len(Z_test_pca)), replace=False)
sc = axes[1].scatter(Z_test_pca[pts, 0], Z_test_pca[pts, 1], c=y_test[pts], cmap="coolwarm",
                     alpha=0.6, edgecolors="none", s=25)
axes[1].set_title(f"2-D PCA Projection of Test Rows by {TASK.upper()} label", fontsize=11, weight="bold")
axes[1].set_xlabel(f"PC1 (top loadings: {pc1_label})")
axes[1].set_ylabel(f"PC2 (top loadings: {pc2_label})")
cbar = plt.colorbar(sc, ax=axes[1], ticks=[0, 1])
cbar.ax.set_yticklabels(["Label 0", "Label 1 (outbreak)"])
plt.tight_layout(); plt.savefig("review3_pca_and_classifiers.png", dpi=300); plt.show()
print("Saved 'review3_pca_and_classifiers.png'")

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, roc_curve,
                             average_precision_score)
if "Z_test" not in globals():
    pca_scaler = StandardScaler().fit(X_train)
    Z_test = pca_scaler.transform(X_test)

# ==============================================================================
# REVIEW III - PART 2: K-MEANS, ELBOW METHOD, HIERARCHICAL CLUSTERING, DENDROGRAM, SILHOUETTE
# Unsupervised: the outbreak label is used ONLY afterwards, to describe what each cluster contains.
# ==============================================================================
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# standardise with the training-world scaler (fitted in Part 1), cluster a sample of the test worlds
rng = np.random.default_rng(SEED)
clust_idx = rng.choice(len(Z_test), size=min(2500, len(Z_test)), replace=False)
X_clust, y_clust = Z_test[clust_idx], y_test[clust_idx]


def elbow_k(ks, inertias):
    """Elbow = point furthest from the straight line joining the first and last (scaled) points."""
    k = np.asarray(ks, float); i = np.asarray(inertias, float)
    k = (k - k.min()) / max(np.ptp(k), 1e-12); i = (i - i.min()) / max(np.ptp(i), 1e-12)
    dx, dy = k[-1] - k[0], i[-1] - i[0]
    dist = np.abs(dx * (i - i[0]) - dy * (k - k[0])) / np.hypot(dx, dy)
    return int(np.asarray(ks)[np.argmax(dist)])


print("=" * 60)
print("3. K-MEANS CLUSTERING & ELBOW METHOD EVALUATION")
print("=" * 60)
K_range = list(range(2, 7))
inertias, silhouette_scores, db_scores = [], [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_clust)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_clust, km.labels_))
    db_scores.append(davies_bouldin_score(X_clust, km.labels_))
    print(f"  K={k} -> Inertia: {km.inertia_:.1f} | Silhouette: {silhouette_scores[-1]:.4f} | "
          f"Davies-Bouldin: {db_scores[-1]:.4f}")

optimal_k = elbow_k(K_range, inertias)
best_sil_k = K_range[int(np.argmax(silhouette_scores))]
print(f"\nElbow method suggests K = {optimal_k}; highest silhouette is at K = {best_sil_k}. Using K = {optimal_k}.")

cluster_eval_df = pd.DataFrame({
    "Clusters (K)": K_range,
    "Inertia (SSE)": [round(x, 1) for x in inertias],
    "Silhouette Score (higher is better)": [round(x, 4) for x in silhouette_scores],
    "Davies-Bouldin Index (lower is better)": [round(x, 4) for x in db_scores],
})
display(cluster_eval_df)
cluster_eval_df.to_csv("review3_cluster_evaluation.csv", index=False)

km_opt = KMeans(n_clusters=optimal_k, random_state=SEED, n_init=10).fit(X_clust)
labels_km = km_opt.labels_
km_sil = silhouette_score(X_clust, labels_km)
km_db = davies_bouldin_score(X_clust, labels_km)
km_inertia = km_opt.inertia_


def cluster_profile_table(labels, centers=None):
    """Describe each cluster by size, true outbreak rate and its most distinctive features."""
    rows = []
    for c in np.unique(labels):
        m = labels == c
        centre = centers[c] if centers is not None else X_clust[m].mean(axis=0)
        top = np.argsort(-np.abs(centre))[:3]
        rows.append({"Cluster": int(c), "Unit-Weeks": int(m.sum()),
                     "Outbreak Positive Rate": round(float(y_clust[m].mean()), 4),
                     "Most distinctive features (z-score)": ", ".join(f"{cls_cols[j]} ({centre[j]:+.2f})" for j in top)})
    table = pd.DataFrame(rows).sort_values("Outbreak Positive Rate").reset_index(drop=True)
    table["Risk rank"] = ["lowest"] + ["middle"] * (len(table) - 2) + ["highest"] if len(table) > 2 else \
        (["lowest", "highest"][:len(table)])
    return table


print("\nK-Means cluster profile vs. true outbreak prevalence (sorted by outbreak rate):")
km_profile = cluster_profile_table(labels_km, km_opt.cluster_centers_)
display(km_profile)
km_profile.to_csv("review3_kmeans_profile.csv", index=False)

# ------------------------------------------------------------------------------
# HIERARCHICAL AGGLOMERATIVE CLUSTERING (Ward linkage)
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("4. HIERARCHICAL AGGLOMERATIVE CLUSTERING (Ward linkage)")
print("=" * 60)
hier = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward").fit(X_clust)
hier_sil = silhouette_score(X_clust, hier.labels_)
hier_db = davies_bouldin_score(X_clust, hier.labels_)
print(f"  K={optimal_k}: silhouette = {hier_sil:.4f}, Davies-Bouldin = {hier_db:.4f}")
hier_profile = cluster_profile_table(hier.labels_)
display(hier_profile)
hier_profile.to_csv("review3_hierarchical_profile.csv", index=False)

# dendrogram on a smaller sample so it stays readable
dend_idx = rng.choice(len(X_clust), size=min(300, len(X_clust)), replace=False)
linked = linkage(X_clust[dend_idx], method="ward")
cut_labels = fcluster(linked, t=optimal_k, criterion="maxclust")
print(f"  Dendrogram sample cut into {len(np.unique(cut_labels))} groups: sizes {np.bincount(cut_labels)[1:].tolist()}")

# ------------------------------------------------------------------------------
# PLOTS: elbow, silhouette, K-means in PCA space, dendrogram
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes[0, 0].plot(K_range, inertias, "bo-", lw=2, markersize=7)
axes[0, 0].axvline(optimal_k, color="red", linestyle="--", label=f"Elbow choice (K={optimal_k})")
axes[0, 0].set_title("Elbow Method: Optimal K Selection", fontsize=11, weight="bold")
axes[0, 0].set_xlabel("Number of Clusters (K)"); axes[0, 0].set_ylabel("Inertia (SSE)"); axes[0, 0].legend()

axes[0, 1].bar(K_range, silhouette_scores, color="#2ca02c", alpha=0.7)
axes[0, 1].set_title("Cluster Evaluation: Silhouette Scores", fontsize=11, weight="bold")
axes[0, 1].set_xlabel("Number of Clusters (K)"); axes[0, 1].set_ylabel("Silhouette Coefficient")

pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_clust)
scat = axes[1, 0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=labels_km, cmap="viridis", alpha=0.7, s=28)
axes[1, 0].set_title(f"K-Means Clusters in 2-D PCA Space (K={optimal_k})", fontsize=11, weight="bold")
axes[1, 0].set_xlabel("Principal Component 1"); axes[1, 0].set_ylabel("Principal Component 2")
plt.colorbar(scat, ax=axes[1, 0], label="Cluster ID")

dendrogram(linked, truncate_mode="lastp", p=30, orientation="top", distance_sort="descending",
           show_leaf_counts=True, ax=axes[1, 1])
axes[1, 1].set_title("Hierarchical Clustering: Ward Dendrogram (last 30 merges)", fontsize=11, weight="bold")
axes[1, 1].set_xlabel("Cluster size (or sample index)"); axes[1, 1].set_ylabel("Ward distance")
plt.tight_layout(); plt.savefig("review3_clustering_elbow_dendrogram.png", dpi=300); plt.show()
print("Saved 'review3_clustering_elbow_dendrogram.png'")

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, roc_curve,
                             average_precision_score)

# ==============================================================================
# REVIEW III - PART 3: DECISION TREES (GINI / ENTROPY), RANDOM FOREST (BAGGING),
#                      SINGLE-LAYER PERCEPTRON, ARTIFICIAL NEURAL NETWORK (MLP)
# Same protocol as before: fit on train worlds, F1 threshold from validation worlds, report on test worlds.
# ==============================================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_curve

part3_models = {
    "Decision Tree (Gini Impurity)": DecisionTreeClassifier(
        criterion="gini", max_depth=6, class_weight="balanced", random_state=SEED),
    "Decision Tree (Information Gain / Entropy)": DecisionTreeClassifier(
        criterion="entropy", max_depth=6, class_weight="balanced", random_state=SEED),
    "Random Forest (Bagging)": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=5, class_weight="balanced",
        random_state=SEED, n_jobs=-1),
    # a perceptron is a single linear threshold unit: it has no probabilities, so its raw decision score is used
    "Single-Layer Perceptron": make_pipeline(StandardScaler(), Perceptron(
        class_weight="balanced", max_iter=1000, tol=1e-3, random_state=SEED)),
    "Artificial Neural Network (MLP 64-32)": make_pipeline(StandardScaler(), MLPClassifier(
        hidden_layer_sizes=(64, 32), activation="relu", early_stopping=True,
        max_iter=300, random_state=SEED)),
}

print("=" * 60)
print("5. TREE-BASED MODELS & NEURAL NETWORKS")
print("=" * 60)
part3_rows, part3_scores = [], {}
for name, model in part3_models.items():
    print(f"  Fitting {name}...")
    row, s_te, _ = evaluate_classifier(name, model, X_train, y_train, X_val, y_val, X_test, y_test)
    part3_rows.append(row)
    part3_scores[name] = s_te

part3_df = pd.DataFrame(part3_rows).rename(columns={"Model": "Model Architecture"})
display(part3_df)
part3_df.to_csv("review3_part3_trees_nn.csv", index=False)

# ------------------------------------------------------------------------------
# PLOTS: ROC curves and the features each tree model relies on most
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, scores in part3_scores.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={safe_auc(y_test, scores):.2f})")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC Curves: Trees & Neural Networks (Test Worlds)", fontsize=11, weight="bold")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].legend(fontsize=8, loc="lower right")

forest = part3_models["Random Forest (Bagging)"]
top = np.argsort(forest.feature_importances_)[::-1][:10][::-1]
axes[1].barh([cls_cols[i] for i in top], forest.feature_importances_[top], color="#1f77b4")
axes[1].set_title("Random Forest: Top-10 Feature Importances", fontsize=11, weight="bold")
axes[1].set_xlabel("Importance")
plt.tight_layout(); plt.savefig("review3_trees_neural_networks.png", dpi=300); plt.show()
print("Saved 'review3_trees_neural_networks.png'")

In [ ]:
# --- Self-contained cell: loads the generated data itself, so it also runs on its own ---
if "X_train" not in globals():
    # ==============================================================================
    # 3. LOAD THE GENERATED WORLDS (shared by every analysis cell below)
    #    Split strictly by independent world: train worlds / validation worlds / test worlds
    # ==============================================================================
    import glob, os, sys, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                 roc_auc_score, precision_recall_curve)

    warnings.filterwarnings("ignore", category=FutureWarning)
    sys.path.insert(0, os.getcwd())                      # make `sanjeevani_ml` importable
    from sanjeevani_ml.pipeline import FEATURE_COLUMNS   # the official feature list

    SEED = 42
    np.random.seed(SEED)

    # Which task to analyse:
    #   "m2" = current detection (is an outbreak active now?)       <- Review II / III default
    #   "m1" = early warning (will a new outbreak start in 7 days?) <- units with no active event only
    TASK = "m2"
    LABEL, RISK_SET = {"m2": ("label_m2_active", None),
                       "m1": ("label_m1_onset_7d", "in_m1_risk_set")}[TASK]


    def find_run_dir():
        """Use the run created by cell 2; otherwise fall back to the newest run on disk."""
        if "RUN_DIR" in globals() and os.path.isdir(os.path.join(RUN_DIR, "ml_datasets")):
            return RUN_DIR
        candidates = [p for p in glob.glob("runs/colab-*") + glob.glob("runs/full-*")
                      if os.path.isdir(os.path.join(p, "ml_datasets"))]
        if not candidates:
            raise FileNotFoundError("No simulation run found. Run cell 2 (simulation) first.")
        return max(candidates, key=os.path.getmtime)


    latest_run = find_run_dir()
    # cohort files only (events_*.csv.gz sit in the same folders and must not be mixed in)
    data_files = sorted(glob.glob(f"{latest_run}/ml_datasets/worlds/*/world_*_*.csv.gz"))
    if not data_files:
        raise FileNotFoundError(f"No world cohort files found under {latest_run}")
    print(f"Loading data from: {latest_run} ({len(data_files)} files)")

    df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
    if RISK_SET:
        df = df[df[RISK_SET] == 1]
    df = df[df[LABEL].notna()].copy().reset_index(drop=True)
    df[LABEL] = df[LABEL].astype(int)

    # In the full study the three splits hold hundreds of thousands of rows each. The classical models in the cells below
    # (KNN, SVM, forests, cross-validation) would take many hours on Colab, so each split is randomly thinned to a cap.
    # The smoke study is never thinned. The clustering cells and the official benchmark always use every row.
    MAX_ROWS_PER_SPLIT = {"smoke": None, "full": 150_000}[globals().get("STUDY", "smoke")]
    if MAX_ROWS_PER_SPLIT:
        keep = []
        for _split in ("train", "val", "test"):
            _idx = df.index[df["split"] == _split]
            if len(_idx) > MAX_ROWS_PER_SPLIT:
                _idx = pd.Index(np.random.default_rng(SEED).choice(_idx, MAX_ROWS_PER_SPLIT, replace=False))
            keep.append(_idx)
        df = df.loc[keep[0].append(keep[1]).append(keep[2]).sort_values()].reset_index(drop=True)
        print(f"Full study: each split thinned to at most {MAX_ROWS_PER_SPLIT:,} rows for the classical models.")

    missing_splits = {"train", "val", "test"} - set(df["split"].unique())
    if missing_splits:
        raise ValueError(f"Missing world splits {sorted(missing_splits)}: refusing to split rows "
                         "randomly, because that would leak a world's truth across train and test.")

    cls_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
    train_mask = (df["split"] == "train").to_numpy()
    val_mask   = (df["split"] == "val").to_numpy()
    test_mask  = (df["split"] == "test").to_numpy()

    imputer = SimpleImputer(strategy="median").fit(df.loc[train_mask, cls_cols])   # fitted on train only
    def _prep(mask):
        return pd.DataFrame(imputer.transform(df.loc[mask, cls_cols]), columns=cls_cols)

    X_train, X_val, X_test = _prep(train_mask), _prep(val_mask), _prep(test_mask)
    y_train = df.loc[train_mask, LABEL].to_numpy()
    y_val   = df.loc[val_mask,   LABEL].to_numpy()
    y_test  = df.loc[test_mask,  LABEL].to_numpy()

    # week of each row: used so CV folds / bootstraps keep a whole week (and all of its
    # surveillance-scenario copies) together instead of splitting near-duplicate rows
    week_train = df.loc[train_mask, "date"].to_numpy()
    week_test  = df.loc[test_mask,  "date"].to_numpy()


    # ---------------- shared helpers ----------------
    def best_f1_threshold(y_true, scores, default=0.5):
        """Threshold that maximises F1 on the supplied (validation) data."""
        if len(np.unique(y_true)) < 2:
            return default
        p, r, t = precision_recall_curve(y_true, scores)
        f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
        return float(t[np.argmax(f1)]) if len(t) else default


    def safe_auc(y_true, scores):
        return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else float("nan")


    def model_scores(model, X):
        """Probability of the positive class, or the raw decision score for models without one."""
        return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)


    def evaluate_classifier(name, model, X_fit, y_fit, X_va, y_va, X_te, y_te):
        """Fit on train, choose the F1 threshold on validation worlds, report on the test worlds."""
        model.fit(X_fit, y_fit)
        thr = best_f1_threshold(y_va, model_scores(model, X_va))
        s_te = model_scores(model, X_te)
        pred = (s_te >= thr).astype(int)
        row = {"Model": name,
               "Test Accuracy":  round(accuracy_score(y_te, pred), 4),
               "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
               "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
               "Test F1-Score":  round(f1_score(y_te, pred, zero_division=0), 4),
               "Test ROC-AUC":   round(safe_auc(y_te, s_te), 4),
               "Operating Threshold": round(thr, 3)}
        return row, s_te, pred


    print(f"Task: {TASK.upper()} ({LABEL}) | features: {len(cls_cols)}")
    print(f"Splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Positive rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

# ==============================================================================
# REVIEW III - PART 4: UNSUPERVISED CLUSTERING FOR SANJEEVANI  (setup + fitting)
# Six clusterings used by the citizen, ASHA worker and hospital dashboards. They describe areas, weeks and events;
# they sit NEXT TO the M1 / M2 classifiers and never replace them. Everything is fitted on the TRAINING worlds only
# and then assigned to the validation / test worlds. Outcome labels are used only to describe and check clusters.
# ==============================================================================
import importlib, time
from pathlib import Path

CLUSTERING_SOURCE = r'''"""Unsupervised clustering for Sanjeevani (synthetic worlds only).

Clustering is a *descriptive* layer next to the M1 / M2 classifiers: it groups areas, weeks and events so that
people can see what kind of situation they are looking at. It never replaces the supervised models, and it never
groups people (the model scores areas, not individuals).

Six clusterings, all fitted on TRAINING worlds only and then assigned to validation / test worlds:

1. ``archetypes``  areas with a similar static profile (size, density, coast, water-source mix, sanitation)
2. ``regimes``     "what kind of week is this": weekly water, weather, case and sensor signals
3. ``blind_spots`` how well each area is watched (sensor uptime, lab recency, report completeness)
4. ``seasonal``    areas whose OPD workload follows the same yearly shape
5. ``event_types`` kinds of simulated outbreak (duration, size, cholera share, reporting delay)
6. ``hotspots``    adjacent flagged areas, found per week with DBSCAN on neighbour-graph distance

The outcome labels are used only to *describe* and *check* clusters afterwards (outbreak rate per cluster, detection
delay per blind-spot cluster). They are never an input to any clustering.

    python -m sanjeevani_ml.clustering --run-dir runs/<run-id>
"""

from __future__ import annotations

import argparse
import json
import math
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# (column, transform). "rate_log" = per 100,000 residents, then log1p; "log" = log1p.
REGIME_FEATURES: list[tuple[str, str | None]] = [
    ("turbidity_mean_7d", None), ("turbidity_exceed_7d", None),
    ("chlorine_mean_7d", None), ("chlorine_depleted_7d", None), ("tds_mean_7d", None),
    ("ecoli_last_cfu", "log"), ("ecoli_positive_30d", None),
    ("rain_7d", "log"), ("rain_30d", "log"), ("heavy_rain_after_dry_7d", None), ("consec_dry_days", "log"),
    ("temp_mean_7d", None), ("rel_humidity_7d", None),
    ("asha_cases_7d", "rate_log"), ("asha_cases_14d", "rate_log"), ("asha_alert_7d", None),
    ("opd_cases_7d", "rate_log"), ("idsp_alerts_28d", None),
    ("sensor_uptime_7d", None),
]

# phrase used when a feature is clearly above / below average in a cluster
REGIME_PHRASES: dict[str, tuple[str | None, str | None]] = {
    "turbidity_mean_7d": ("turbid water", None), "turbidity_exceed_7d": ("turbid water", None),
    "chlorine_mean_7d": (None, "low chlorine"), "chlorine_depleted_7d": ("low chlorine", None),
    "tds_mean_7d": ("high dissolved solids", None),
    "ecoli_last_cfu": ("E. coli positives", None), "ecoli_positive_30d": ("E. coli positives", None),
    "rain_7d": ("heavy rain", "dry week"), "rain_30d": ("wet month", "dry month"),
    "heavy_rain_after_dry_7d": ("rain after a dry spell", None), "consec_dry_days": ("dry spell", None),
    "temp_mean_7d": ("hot weather", "cool weather"), "rel_humidity_7d": ("humid", None),
    "asha_cases_7d": ("case surge", None), "asha_cases_14d": ("case surge", None), "asha_alert_7d": ("ASHA cluster alert", None),
    "opd_cases_7d": ("high OPD load", None), "idsp_alerts_28d": ("IDSP alerts", None),
    "sensor_uptime_7d": (None, "sensors offline"),
}

ARCHETYPE_PHRASES: dict[str, tuple[str | None, str | None]] = {
    "log_population": ("large population", "small population"), "log_density": ("dense", "sparse"),
    "coastal": ("coastal", None), "share_piped": ("piped-water reliant", "little piped water"),
    "share_groundwater": ("groundwater reliant", None), "share_tanker": ("tanker dependent", None),
    "share_surface": ("surface-water reliant", None), "sanitation_coverage": ("high sanitation", "low sanitation"),
}


# ───────────────────────────── helpers ─────────────────────────────
def _clean(value):
    """JSON-safe scalar: NaN / inf become None, numpy types become Python types."""
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return None if (math.isnan(value) or math.isinf(value)) else value
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def _round(value, digits=4):
    value = _clean(value)
    return round(value, digits) if isinstance(value, float) else value


def choose_k(X: np.ndarray, ks: range, seed: int = SEED, sample: int = 4000) -> tuple[int, list[dict]]:
    """Pick K by silhouette plus split-half stability (adjusted Rand index between two models fitted on halves).

    Stability matters more than the elbow here: weekly signals form overlapping clouds, so a K that changes
    from one random half to the next is not a real structure.
    """
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=min(sample, len(X)), replace=False)
    Xs = X[idx]
    rows = []
    for k in ks:
        if k >= len(Xs):
            continue
        labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Xs)
        sil = float(silhouette_score(Xs, labels)) if len(set(labels)) > 1 else 0.0
        aris = []
        for rep in range(4):
            perm = np.random.default_rng(seed + rep).permutation(len(Xs))
            half = len(Xs) // 2
            a = KMeans(n_clusters=k, n_init=5, random_state=seed + rep).fit(Xs[perm[:half]])
            b = KMeans(n_clusters=k, n_init=5, random_state=seed + 100 + rep).fit(Xs[perm[half:]])
            aris.append(adjusted_rand_score(a.predict(Xs), b.predict(Xs)))
        stab = float(np.mean(aris))
        rows.append({"k": k, "silhouette": _round(sil), "stability": _round(stab), "score": _round((sil + stab) / 2)})
    best = max(rows, key=lambda r: r["score"])["k"]
    return best, rows


def describe_center(center: np.ndarray, names: list[str], phrases: dict, threshold: float = 0.5,
                    top: int = 2, default: str = "Typical") -> tuple[str, list[dict]]:
    """Name a cluster from the features that stand furthest from average (z-scores)."""
    contributions = []
    for name, z in zip(names, center):
        pos, neg = phrases.get(name, (None, None))
        phrase = pos if z > 0 else neg
        if phrase and abs(z) >= threshold:
            contributions.append((abs(float(z)), phrase, name, float(z)))
    contributions.sort(reverse=True)
    seen, chosen = set(), []
    for _, phrase, _, _ in contributions:
        if phrase not in seen:
            seen.add(phrase)
            chosen.append(phrase)
        if len(chosen) == top:
            break
    label = " + ".join(chosen).capitalize() if chosen else default
    signals = [{"feature": f, "z": _round(z, 2)} for _, _, f, z in contributions[:4]]
    return label, signals


def _units_frame(units: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(units).set_index("unit_id")


def load_units_yaml(path: Path) -> list[dict]:
    import yaml
    raw = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
    return [{"unit_id": u.get("id"), **{k: v for k, v in u.items() if k != "id"}} for u in raw.get("units", [])]


# ───────────────────────────── data ─────────────────────────────
def _read_split(worlds_dir: Path, split: str, columns: list[str] | None) -> pd.DataFrame:
    frames = []
    for path in sorted((worlds_dir / split).glob("world_*_*.csv.gz")):
        frames.append(pd.read_csv(path, usecols=columns))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_run_tables(run_dir: Path) -> dict:
    """Cohort and event tables of a completed run (reads only what the clusterings need)."""
    worlds = Path(run_dir) / "ml_datasets" / "worlds"
    wanted = sorted({"world_id", "scenario", "unit_id", "date", "month", "in_m1_risk_set",
                     "label_m1_onset_7d", "label_m2_active", "days_since_lab_test"}
                    | {c for c, _ in REGIME_FEATURES})
    tables = {split: _read_split(worlds, split, wanted) for split in ("train", "val", "test")}
    events = {}
    for split in ("train", "val", "test"):
        paths = sorted((worlds / split).glob("events_*.csv.gz"))
        events[split] = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True) if paths else pd.DataFrame()
    return {"cohorts": tables, "events": events}


# ───────────────────────────── 1. archetypes ─────────────────────────────
def _archetype_matrix(units: list[dict]) -> tuple[np.ndarray, list[str], list[str]]:
    frame = _units_frame(units)
    sources = sorted({k for u in units for k in (u.get("water_sources") or {})})
    cols = {
        "log_population": np.log(frame["population"].astype(float)),
        "log_density": np.log(frame["density_km2"].astype(float)),
        "coastal": frame["coastal"].astype(float),
        "sanitation_coverage": frame["sanitation_coverage"].astype(float),
    }
    for s in sources:
        cols[f"share_{s}"] = frame["water_sources"].apply(lambda d, s=s: float((d or {}).get(s, 0.0)))
    matrix = pd.DataFrame(cols, index=frame.index)
    return matrix.to_numpy(float), list(matrix.columns), list(matrix.index)


def fit_archetypes(units: list[dict], seed: int = SEED) -> dict:
    X, names, ids = _archetype_matrix(units)
    scaler = StandardScaler().fit(X)
    Z = scaler.transform(X)
    k, diagnostics = choose_k(Z, range(3, 6), seed, sample=len(Z))
    km = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(Z)
    labels = km.labels_
    clusters = []
    for c in range(k):
        name, signals = describe_center(km.cluster_centers_[c], names, ARCHETYPE_PHRASES, 0.45, 2, "Typical unit")
        members = [ids[i] for i in np.flatnonzero(labels == c)]
        clusters.append({"id": c, "name": name, "size": len(members), "members": members, "signals": signals})
    return {"k": k, "diagnostics": diagnostics, "clusters": clusters,
            "unit_cluster": {uid: int(l) for uid, l in zip(ids, labels)},
            "quality": {"silhouette": _round(silhouette_score(Z, labels))}}


# ───────────────────────────── 2. weekly regimes ─────────────────────────────
def _regime_matrix(df: pd.DataFrame, population: dict[str, float]) -> np.ndarray:
    out = {}
    pop = df["unit_id"].map(population).astype(float)
    for col, tf in REGIME_FEATURES:
        s = df[col].astype(float)
        if tf == "log":
            s = np.log1p(s.clip(lower=0))
        elif tf == "rate_log":
            s = np.log1p((s.clip(lower=0) / pop) * 1e5)
        out[col] = s
    return pd.DataFrame(out).to_numpy(float)


def fit_regimes(train: pd.DataFrame, population: dict[str, float], seed: int = SEED) -> dict:
    names = [c for c, _ in REGIME_FEATURES]
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    Z = scaler.fit_transform(imputer.fit_transform(_regime_matrix(train, population)))
    k, diagnostics = choose_k(Z, range(3, 7), seed)
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Z)
    # order regimes from calmest to most outbreak-heavy, using the TRAIN outcome only to order and describe them
    rates = pd.Series(train["label_m2_active"].to_numpy(float)).groupby(km.labels_).mean()
    order = list(rates.sort_values().index)
    remap = {old: new for new, old in enumerate(order)}
    centers = km.cluster_centers_[order]
    labels = []
    for c in range(k):
        name, signals = describe_center(centers[c], names, REGIME_PHRASES, 0.5, 2, "Quiet baseline")
        labels.append((name, signals))
    return {"imputer": imputer, "scaler": scaler, "km": km, "names": names, "remap": remap, "k": k,
            "diagnostics": diagnostics, "labels": labels, "population": population,
            "silhouette": _round(silhouette_score(Z[:5000], km.labels_[:5000]))}


def assign_regimes(model: dict, df: pd.DataFrame) -> np.ndarray:
    Z = model["scaler"].transform(model["imputer"].transform(_regime_matrix(df, model["population"])))
    return np.array([model["remap"][int(l)] for l in model["km"].predict(Z)])


def regime_report(model: dict, test: pd.DataFrame, assigned: np.ndarray) -> dict:
    base_m2 = float(test["label_m2_active"].mean())
    risk = test["in_m1_risk_set"] == 1
    base_m1 = float(test.loc[risk, "label_m1_onset_7d"].mean()) if risk.any() else None
    clusters = []
    for c in range(model["k"]):
        mask = assigned == c
        sub = test.loc[mask]
        sub_risk = sub["in_m1_risk_set"] == 1
        m2 = float(sub["label_m2_active"].mean()) if len(sub) else None
        m1 = float(sub.loc[sub_risk, "label_m1_onset_7d"].mean()) if sub_risk.any() else None
        name, signals = model["labels"][c]
        clusters.append({
            "id": c, "name": name, "signals": signals, "share": _round(mask.mean()),
            "unit_weeks": int(mask.sum()), "m2_active_rate": _round(m2), "m1_onset_rate": _round(m1),
            "m2_lift": _round(m2 / base_m2) if (m2 is not None and base_m2) else None,
            "m1_lift": _round(m1 / base_m1) if (m1 is not None and base_m1) else None,
        })
    return {"k": model["k"], "diagnostics": model["diagnostics"], "silhouette": model["silhouette"],
            "baseline": {"m2_active_rate": _round(base_m2), "m1_onset_rate": _round(base_m1)}, "clusters": clusters}


# ───────────────────────────── 3. surveillance blind spots ─────────────────────────────
BLIND_NAMES = ["Well covered", "Partly covered", "Blind spot"]


def _coverage_table(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["lab_gap"] = work["days_since_lab_test"].fillna(work["days_since_lab_test"].max())
    grouped = work.groupby(["world_id", "scenario", "unit_id"])
    table = grouped.agg(uptime=("sensor_uptime_7d", "mean"), lab_gap=("lab_gap", "mean"),
                        asha=("asha_cases_7d", "sum"), opd=("opd_cases_7d", "sum")).reset_index()
    table["report_ratio"] = table["asha"] / (table["opd"] + 1.0)
    return table[["world_id", "scenario", "unit_id", "uptime", "lab_gap", "report_ratio"]]


def fit_blind_spots(train: pd.DataFrame, seed: int = SEED) -> dict:
    table = _coverage_table(train)
    cols = ["uptime", "lab_gap", "report_ratio"]
    scaler = StandardScaler().fit(table[cols])
    Z = scaler.transform(table[cols])
    km = KMeans(n_clusters=3, n_init=20, random_state=seed).fit(Z)
    blindness = -km.cluster_centers_[:, 0] + km.cluster_centers_[:, 1] - km.cluster_centers_[:, 2]
    order = list(np.argsort(blindness))
    return {"scaler": scaler, "km": km, "cols": cols, "remap": {old: new for new, old in enumerate(order)},
            "silhouette": _round(silhouette_score(Z, km.labels_)) if len(Z) > 3 else None,
            "centers": {BLIND_NAMES[new]: {c: _round(v) for c, v in zip(cols, km.cluster_centers_[old] * scaler.scale_ + scaler.mean_)}
                        for new, old in enumerate(order)}}


def assign_blind_spots(model: dict, df: pd.DataFrame) -> pd.DataFrame:
    table = _coverage_table(df)
    Z = model["scaler"].transform(table[model["cols"]])
    table["cluster"] = [model["remap"][int(l)] for l in model["km"].predict(Z)]
    return table


def blind_spot_report(model: dict, test: pd.DataFrame, events: pd.DataFrame) -> dict:
    assigned = assign_blind_spots(model, test)
    lookup = {(r.world_id, r.scenario, r.unit_id): int(r.cluster) for r in assigned.itertuples()}
    rows = []
    if len(events):
        start = pd.to_datetime(events["operational_start_date"])
        for scenario in sorted(test["scenario"].unique()):
            col = f"first_report_date_{scenario}"
            if col not in events:
                continue
            delay = (pd.to_datetime(events[col]) - start).dt.days
            for ev, d in zip(events.itertuples(), delay):
                cluster = lookup.get((ev.world_id, scenario, ev.unit_id))
                if cluster is not None:
                    rows.append({"cluster": cluster, "delay": None if pd.isna(d) else float(d)})
    frame = pd.DataFrame(rows, columns=["cluster", "delay"])
    clusters = []
    for c, name in enumerate(BLIND_NAMES):
        sub = frame[frame["cluster"] == c]
        reported = sub["delay"].dropna()
        members = assigned[assigned["cluster"] == c]
        clusters.append({
            "id": c, "name": name, "unit_scenarios": int(len(members)),
            "events": int(len(sub)),
            "unreported_share": _round(float(sub["delay"].isna().mean())) if len(sub) else None,
            "median_report_delay_days": _round(float(reported.median())) if len(reported) else None,
            "mean_sensor_uptime": _round(float(members["uptime"].mean())) if len(members) else None,
            "mean_days_since_lab_test": _round(float(members["lab_gap"].mean())) if len(members) else None,
        })
    unit_cluster: dict[str, dict[str, dict[str, int]]] = {}
    for (world, scenario, unit), c in lookup.items():
        unit_cluster.setdefault(world, {}).setdefault(scenario, {})[unit] = c
    return {"clusters": clusters, "centers": model["centers"], "silhouette": model["silhouette"],
            "unit_cluster": unit_cluster}


# ───────────────────────────── 4. seasonal trajectories ─────────────────────────────
def fit_seasonal(train: pd.DataFrame, population: dict[str, float], seed: int = SEED) -> dict:
    work = train.copy()
    work["rate"] = work["opd_cases_7d"] / work["unit_id"].map(population).astype(float) * 1e5
    monthly = work.groupby(["unit_id", "month"])["rate"].mean().unstack("month").reindex(columns=range(1, 13))
    monthly = monthly.apply(lambda row: row.fillna(row.mean()), axis=1)
    index = monthly.div(monthly.mean(axis=1), axis=0)          # 1.0 = that unit's own yearly average
    Z = index.to_numpy(float)
    k, diagnostics = choose_k(Z, range(2, 5), seed, sample=len(Z))
    km = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(Z)
    clusters = []
    for c in range(k):
        curve = km.cluster_centers_[c]
        amplitude = float(curve.max() - curve.min())
        peak = MONTHS[int(curve.argmax())]
        strength = "Strong" if amplitude >= 0.8 else "Moderate" if amplitude >= 0.3 else "Mild"
        name = "Flat year-round" if amplitude < 0.15 else f"{strength} seasonal peak around {peak}"
        members = [uid for uid, l in zip(index.index, km.labels_) if l == c]
        clusters.append({"id": c, "name": name, "size": len(members), "members": members,
                         "peak_month": peak, "amplitude": _round(amplitude), "curve": [_round(v, 3) for v in curve]})
    return {"k": k, "diagnostics": diagnostics, "clusters": clusters, "months": MONTHS,
            "unit_cluster": {uid: int(l) for uid, l in zip(index.index, km.labels_)},
            "quality": {"silhouette": _round(silhouette_score(Z, km.labels_))}}


# ───────────────────────────── 5. outbreak event types ─────────────────────────────
def _event_matrix(events: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    start = pd.to_datetime(events["operational_start_date"])
    report = pd.to_datetime(events["first_report_date_baseline"])
    delay = (report - start).dt.days
    unreported = delay.isna()
    delay_filled = delay.fillna(events["duration_days"])
    total = events["total_cases"].astype(float)
    # Only properties of the outbreak itself are clustered. Whether and how fast it was reported is a property of
    # the surveillance system, so it is described per cluster afterwards instead of driving the grouping.
    frame = pd.DataFrame({
        "log_duration": np.log1p(events["duration_days"].astype(float)),
        "log_cases": np.log1p(total),
        "cholera_share": events["cholera_cases"].astype(float) / total.clip(lower=1),
    })
    detail = pd.DataFrame({"duration": events["duration_days"], "cases": total, "cholera_share": frame["cholera_share"],
                           "unreported": unreported, "delay": delay})
    return frame.to_numpy(float), detail


def fit_event_types(train_events: pd.DataFrame, seed: int = SEED) -> dict | None:
    if len(train_events) < 12:
        return None
    X, detail = _event_matrix(train_events)
    scaler = StandardScaler().fit(X)
    Z = scaler.transform(X)
    k, diagnostics = choose_k(Z, range(2, 5), seed, sample=len(Z))
    km = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(Z)
    order = list(detail.groupby(km.labels_)["cases"].median().sort_values().index)
    remap = {old: new for new, old in enumerate(order)}
    med_dur, med_cases = float(detail["duration"].median()), float(detail["cases"].median())
    names = []
    for old in order:
        sub = detail[km.labels_ == old]
        dur, cases, unrep = float(sub["duration"].median()), float(sub["cases"].median()), float(sub["unreported"].mean())
        d = "Long" if dur >= med_dur * 1.15 else "Short" if dur <= med_dur * 0.85 else "Typical-length"
        s = "large" if cases >= med_cases * 1.15 else "small" if cases <= med_cases * 0.85 else "mid-size"
        names.append(f"{d}, {s}")
    return {"scaler": scaler, "km": km, "remap": remap, "names": names, "k": k, "diagnostics": diagnostics,
            "silhouette": _round(silhouette_score(Z, km.labels_))}


def event_type_report(model: dict | None, events: pd.DataFrame) -> dict:
    if model is None or not len(events):
        return {"available": False, "clusters": []}
    X, detail = _event_matrix(events)
    labels = np.array([model["remap"][int(l)] for l in model["km"].predict(model["scaler"].transform(X))])
    clusters = []
    for c in range(model["k"]):
        sub = detail[labels == c]
        reported = sub["delay"].dropna()
        clusters.append({
            "id": c, "name": model["names"][c], "events": int(len(sub)),
            "share": _round(len(sub) / len(detail)),
            "median_duration_days": _round(float(sub["duration"].median())) if len(sub) else None,
            "median_total_cases": _round(float(sub["cases"].median())) if len(sub) else None,
            "mean_cholera_share": _round(float(sub["cholera_share"].mean())) if len(sub) else None,
            "unreported_share": _round(float(sub["unreported"].mean())) if len(sub) else None,
            "median_report_delay_days": _round(float(reported.median())) if len(reported) else None,
        })
    return {"available": True, "k": model["k"], "diagnostics": model["diagnostics"],
            "silhouette": model["silhouette"], "events": int(len(detail)), "clusters": clusters}


# ───────────────────────────── 6. hotspots (per week) ─────────────────────────────
def _adjacency(units: list[dict]) -> dict[str, set[str]]:
    adj: dict[str, set[str]] = {u["unit_id"]: set() for u in units}
    for u in units:
        for n in (u.get("neighbors") or []):
            if n in adj:
                adj[u["unit_id"]].add(n)
                adj[n].add(u["unit_id"])
    return adj


def _hops(adj: dict[str, set[str]], source: str) -> dict[str, int]:
    seen, queue = {source: 0}, deque([source])
    while queue:
        cur = queue.popleft()
        for nxt in adj.get(cur, ()):
            if nxt not in seen:
                seen[nxt] = seen[cur] + 1
                queue.append(nxt)
    return seen


def hotspot_clusters(units: list[dict], board_units: list[dict], task: str,
                     tiers: tuple[str, ...] = ("review",)) -> list[dict]:
    """Groups of adjacent Review-tier areas for one week (DBSCAN on neighbour-graph hop distance).

    Only the Review tier is used: including Watch units chains most of the state into one blob.
    """
    adj = _adjacency(units)
    flagged = [u for u in board_units if (u.get(task) or {}).get("tier") in tiers]
    if len(flagged) < 2:
        return []
    ids = [u["unit_id"] for u in flagged]
    far = 99.0
    dist = np.full((len(ids), len(ids)), far)
    for i, a in enumerate(ids):
        hops = _hops(adj, a)
        for j, b in enumerate(ids):
            dist[i, j] = float(hops.get(b, far))
    labels = DBSCAN(eps=1.0, min_samples=2, metric="precomputed").fit_predict(dist)
    groups = []
    for label in sorted(set(labels) - {-1}):
        members = [flagged[i] for i in np.flatnonzero(labels == label)]
        members.sort(key=lambda u: (u[task] or {}).get("score") or 0.0, reverse=True)
        groups.append({
            "task": task, "size": len(members),
            "review_count": sum(1 for u in members if u[task]["tier"] == "review"),
            "units": [u["unit_id"] for u in members],
            "top_unit": members[0]["unit_id"], "max_score": _round((members[0][task] or {}).get("score")),
        })
    groups.sort(key=lambda g: (g["review_count"], g["max_score"] or 0.0), reverse=True)
    for i, g in enumerate(groups):
        g["id"] = i
    return groups


# ───────────────────────────── orchestration ─────────────────────────────
class ClusterBundle:
    """Fitted clusterings for one run plus the assignments the UI needs."""

    def __init__(self, units: list[dict], tables: dict, seed: int = SEED):
        population = {u["unit_id"]: float(u["population"]) for u in units}
        train, test = tables["cohorts"]["train"], tables["cohorts"]["test"]
        self.units = units
        self.archetypes = fit_archetypes(units, seed)
        self.regime_model = fit_regimes(train, population, seed)
        self.test_regimes = assign_regimes(self.regime_model, test)
        self.test = test
        self.regimes = regime_report(self.regime_model, test, self.test_regimes)
        self.blind_model = fit_blind_spots(train, seed)
        self.blind_spots = blind_spot_report(self.blind_model, test, tables["events"]["test"])
        self.seasonal = fit_seasonal(train, population, seed)
        event_model = fit_event_types(tables["events"]["train"], seed)
        self.event_types = event_type_report(event_model, tables["events"]["test"])
        # (world, scenario, date) -> {unit: regime}
        self._weekly: dict[tuple[str, str, str], dict[str, int]] = {}
        keyed = test[["world_id", "scenario", "date", "unit_id"]].assign(regime=self.test_regimes)
        for (world, scenario, date), group in keyed.groupby(["world_id", "scenario", "date"]):
            self._weekly[(world, scenario, str(date))] = dict(zip(group["unit_id"], group["regime"].astype(int)))

    def static_report(self) -> dict:
        return {"synthetic": True, "archetypes": self.archetypes, "regimes": self.regimes,
                "blind_spots": {k: v for k, v in self.blind_spots.items() if k != "unit_cluster"},
                "seasonal": self.seasonal, "event_types": self.event_types}

    def week(self, world_id: str, scenario: str, date: str, board_units: list[dict] | None = None) -> dict:
        regimes = self._weekly.get((world_id, scenario, date), {})
        blind = self.blind_spots["unit_cluster"].get(world_id, {}).get(scenario, {})
        hotspots = {}
        if board_units is not None:
            hotspots = {task: hotspot_clusters(self.units, board_units, task) for task in ("m1", "m2")}
        return {"world_id": world_id, "scenario": scenario, "date": date, "unit_regime": regimes,
                "unit_blind_spot": blind, "hotspots": hotspots}


def run(run_dir: Path, units: list[dict], out: Path | None = None) -> dict:
    bundle = ClusterBundle(units, load_run_tables(run_dir))
    report = bundle.static_report()
    if out is not None:
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    return report


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--run-dir", type=Path, required=True)
    parser.add_argument("--units", type=Path, default=Path(__file__).resolve().parent.parent / "config" / "tamil_nadu_districts.yaml")
    parser.add_argument("--out", type=Path, default=None, help="default: <run-dir>/results/cluster_summary.json")
    args = parser.parse_args()
    out = args.out or args.run_dir / "results" / "cluster_summary.json"
    report = run(args.run_dir, load_units_yaml(args.units), out)
    print(f"archetypes K={report['archetypes']['k']} | regimes K={report['regimes']['k']} | "
          f"seasonal K={report['seasonal']['k']} | event types="
          f"{report['event_types'].get('k', 'n/a')}. Wrote {out}")


if __name__ == "__main__":
    main()
'''

MODULE_PATH = Path("sanjeevani_ml") / "clustering.py"
if MODULE_PATH.exists():
    print("Using the project's own module:", MODULE_PATH)
else:
    # the repository cloned by cell 1 predates the clustering module, so write the bundled copy next to it
    MODULE_PATH.write_text(CLUSTERING_SOURCE, encoding="utf-8")
    print("Wrote the bundled clustering module to", MODULE_PATH)
importlib.invalidate_caches()
from sanjeevani_ml import clustering as cl

units = cl.load_units_yaml(Path("config/tamil_nadu_districts.yaml"))
POPULATION = {u["unit_id"]: float(u["population"]) for u in units}

t0 = time.time()
cluster_tables = cl.load_run_tables(Path(latest_run))          # latest_run comes from cell 3
bundle = cl.ClusterBundle(units, cluster_tables)
cluster_report = bundle.static_report()
print(f"Fitted all clusterings on {cluster_tables['cohorts']['train']['world_id'].nunique()} training world(s) "
      f"in {time.time() - t0:.0f}s; assigned {cluster_tables['cohorts']['test']['world_id'].nunique()} test world(s).")
print(f"Units: {len(units)} | train rows: {len(cluster_tables['cohorts']['train'])} | "
      f"test rows: {len(cluster_tables['cohorts']['test'])} | test events: {len(cluster_tables['events']['test'])}")

In [ ]:
if "bundle" not in globals():
    raise RuntimeError("Run the clustering setup cell above first (it fits the six clusterings).")

# ==============================================================================
# CLUSTERING 1 AND 2: AREA ARCHETYPES ("areas like yours") AND WEEKLY REGIMES ("what kind of week is this")
# ==============================================================================
from sklearn.metrics import adjusted_rand_score

arch, reg = cluster_report["archetypes"], cluster_report["regimes"]

# ---- how K was chosen: silhouette AND split-half stability (a K that changes between random halves is not real) ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, title, diag, k in [(axes[0], "Area archetypes (43 units)", arch["diagnostics"], arch["k"]),
                           (axes[1], "Weekly regimes (unit-weeks)", reg["diagnostics"], reg["k"])]:
    ks = [d["k"] for d in diag]
    ax.plot(ks, [d["silhouette"] for d in diag], "o-", lw=2, label="Silhouette")
    ax.plot(ks, [d["stability"] for d in diag], "s-", lw=2, label="Split-half stability (ARI)")
    ax.axvline(k, color="red", ls="--", label=f"Chosen K = {k}")
    ax.set_title(title, fontsize=11, weight="bold"); ax.set_xlabel("Number of clusters (K)"); ax.set_xticks(ks)
    ax.set_ylim(0, 1); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("review3_cluster_k_selection.png", dpi=300); plt.show()

# ---- 1. area archetypes ----
print("=" * 70); print("1. AREA ARCHETYPES (K-means on each unit's static profile)"); print("=" * 70)
arch_df = pd.DataFrame([{"Cluster": c["id"], "Archetype": c["name"], "Units": c["size"],
                         "Members": ", ".join(c["members"][:8]) + (" ..." if c["size"] > 8 else "")} for c in arch["clusters"]])
display(arch_df)
arch_df.to_csv("review3_cluster_archetypes.csv", index=False)
print(f"K = {arch['k']}, silhouette = {arch['quality']['silhouette']:.3f}")

# ---- 2. weekly regimes ----
print("\n" + "=" * 70); print("2. WEEKLY REGIMES (K-means on weekly water, weather, case and sensor signals)"); print("=" * 70)
regime_df = pd.DataFrame([{
    "Regime": c["id"], "Name": c["name"], "Share of unit-weeks": c["share"],
    "M2 active-outbreak rate": c["m2_active_rate"], "M2 lift": c["m2_lift"],
    "M1 new-onset rate (7d)": c["m1_onset_rate"], "M1 lift": c["m1_lift"],
} for c in reg["clusters"]])
display(regime_df)
regime_df.to_csv("review3_cluster_regimes.csv", index=False)
print(f"K = {reg['k']}, silhouette = {reg['silhouette']:.3f}")
print(f"Test-world averages: M2 active rate = {reg['baseline']['m2_active_rate']:.1%}, "
      f"M1 onset rate = {reg['baseline']['m1_onset_rate']:.1%}")

lo, hi = reg["clusters"][0], reg["clusters"][-1]
print(f"\nThe calmest regime ('{lo['name']}') has an outbreak rate of {lo['m2_active_rate']:.1%}; "
      f"the hottest ('{hi['name']}') has {hi['m2_active_rate']:.1%} "
      f"({hi['m2_active_rate'] / max(lo['m2_active_rate'], 1e-9):.1f}x higher). "
      f"New-onset rate: {lo['m1_onset_rate']:.1%} vs {hi['m1_onset_rate']:.1%}.")

# ---- robustness: refit with a different random seed and compare the assignments of the test weeks ----
alt = cl.fit_regimes(cluster_tables["cohorts"]["train"], POPULATION, seed=7)
ari_seed = adjusted_rand_score(bundle.test_regimes, cl.assign_regimes(alt, bundle.test))
print(f"Seed robustness: regimes refitted with another seed agree with these at ARI = {ari_seed:.2f} "
      f"(1.0 = identical grouping; K = {alt['k']} this time).")

fig, ax = plt.subplots(figsize=(11, 4.6))
x = np.arange(len(reg["clusters"])); w = 0.38
ax.bar(x - w / 2, [c["m2_active_rate"] for c in reg["clusters"]], w, color="#E53935", label="M2: outbreak active this week")
ax.bar(x + w / 2, [c["m1_onset_rate"] for c in reg["clusters"]], w, color="#FB8C00", label="M1: new outbreak within 7 days")
ax.axhline(reg["baseline"]["m2_active_rate"], color="#E53935", ls="--", lw=1)
ax.axhline(reg["baseline"]["m1_onset_rate"], color="#FB8C00", ls="--", lw=1)
ax.set_xticks(x); ax.set_xticklabels([c["name"].replace(" + ", "\n+ ") for c in reg["clusters"]], fontsize=8)
ax.set_ylabel("Rate in held-out test weeks"); ax.legend()
ax.set_title("Weekly regimes ordered calm to hot: do they separate real outbreak risk? (dashed = test average)", fontsize=11, weight="bold")
plt.tight_layout(); plt.savefig("review3_cluster_regimes.png", dpi=300); plt.show()
print("Saved review3_cluster_k_selection.png, review3_cluster_regimes.png")

In [ ]:
if "bundle" not in globals():
    raise RuntimeError("Run the clustering setup cell above first (it fits the six clusterings).")

# ==============================================================================
# CLUSTERING 3, 4 AND 5: SURVEILLANCE BLIND SPOTS, SEASONAL PATTERNS, OUTBREAK TYPES
# ==============================================================================
blind, seas, ev = cluster_report["blind_spots"], cluster_report["seasonal"], cluster_report["event_types"]

# ---- 3. surveillance blind spots ----
print("=" * 70); print("3. SURVEILLANCE BLIND SPOTS (K-means, K=3, on sensor uptime / lab recency / report completeness)"); print("=" * 70)
blind_df = pd.DataFrame([{
    "Group": c["name"], "Unit-scenarios": c["unit_scenarios"], "Mean sensor uptime": c["mean_sensor_uptime"],
    "Mean days since lab test": c["mean_days_since_lab_test"], "Simulated events": c["events"],
    "Events never reported": c["unreported_share"], "Median report delay (days)": c["median_report_delay_days"],
} for c in blind["clusters"]])
display(blind_df)
blind_df.to_csv("review3_cluster_blind_spots.csv", index=False)
best, worst = blind["clusters"][0], blind["clusters"][-1]
print(f"Check against hidden truth: events in '{worst['name']}' areas went unreported {worst['unreported_share']:.0%} of the time "
      f"(median delay {worst['median_report_delay_days']} days) vs {best['unreported_share']:.0%} "
      f"({best['median_report_delay_days']} days) in '{best['name']}' areas.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = [c["name"] for c in blind["clusters"]]
axes[0].bar(names, [c["unreported_share"] for c in blind["clusters"]], color=["#43A047", "#FB8C00", "#E53935"])
axes[0].set_title("Share of simulated events never reported", fontsize=11, weight="bold"); axes[0].set_ylim(0, 1)
axes[1].bar(names, [c["median_report_delay_days"] or 0 for c in blind["clusters"]], color=["#43A047", "#FB8C00", "#E53935"])
axes[1].set_title("Median days from outbreak start to first report", fontsize=11, weight="bold")
plt.tight_layout(); plt.savefig("review3_cluster_blind_spots.png", dpi=300); plt.show()

# ---- 4. seasonal patterns ----
print("\n" + "=" * 70); print("4. SEASONAL PATTERNS (K-means on each unit's 12-month OPD profile)"); print("=" * 70)
seasonal_df = pd.DataFrame([{"Cluster": c["id"], "Pattern": c["name"], "Units": c["size"], "Peak month": c["peak_month"],
                             "Amplitude (max - min of monthly index)": c["amplitude"]} for c in seas["clusters"]])
display(seasonal_df)
seasonal_df.to_csv("review3_cluster_seasonal.csv", index=False)
print(f"K = {seas['k']}, silhouette = {seas['quality']['silhouette']:.3f}")
fig, ax = plt.subplots(figsize=(10, 4.2))
for c in seas["clusters"]:
    ax.plot(seas["months"], c["curve"], "o-", lw=2, label=f"{c['name']} ({c['size']} units)")
ax.axhline(1, color="grey", ls="--", lw=1); ax.set_ylabel("OPD visits / unit's own yearly average"); ax.legend()
ax.set_title("Seasonal OPD profile per cluster", fontsize=11, weight="bold")
plt.tight_layout(); plt.savefig("review3_cluster_seasonal.png", dpi=300); plt.show()

# ---- 5. outbreak event types ----
print("\n" + "=" * 70); print("5. OUTBREAK TYPES (K-means on outbreak length, size and cholera share)"); print("=" * 70)
if ev.get("available"):
    event_df = pd.DataFrame([{
        "Type": c["name"], "Events": c["events"], "Share": c["share"], "Median length (days)": c["median_duration_days"],
        "Median total cases": c["median_total_cases"], "Cholera share": c["mean_cholera_share"],
        "Never reported": c["unreported_share"], "Median report delay (days)": c["median_report_delay_days"],
    } for c in ev["clusters"]])
    display(event_df)
    event_df.to_csv("review3_cluster_event_types.csv", index=False)
    print(f"K = {ev['k']}, silhouette = {ev['silhouette']:.3f}, {ev['events']} test events")
    print("Note: in a smoke run almost every simulated event is acute diarrheal disease, so cholera share does not "
          "separate the types; the full multi-world study gives more events and a better test of this clustering.")
else:
    event_df = pd.DataFrame()
    print("Too few simulated outbreaks in the training worlds to cluster them.")

In [ ]:
if "bundle" not in globals():
    raise RuntimeError("Run the clustering setup cell above first (it fits the six clusterings).")

# ==============================================================================
# CLUSTERING 6: HOTSPOTS - groups of ADJACENT units that are in the Review tier in the same week
# DBSCAN on neighbour-graph distance (eps = 1 hop, min 2 units), built from the saved model alerts of the test worlds.
# ==============================================================================
res_dir = Path(latest_run) / "results"
pred = {t: pd.read_csv(res_dir / f"test_predictions_{t}.csv.gz") for t in ("m1", "m2")}
hot_world = sorted(pred["m2"]["world_id"].unique())[0]
hot_scenario = sorted(pred["m2"]["scenario"].unique())[0]
sel = {t: pred[t][(pred[t]["world_id"] == hot_world) & (pred[t]["scenario"] == hot_scenario)].copy() for t in pred}
for t in sel:
    sel[t]["tier"] = np.where(sel[t]["alert_f1"].astype(str).str.lower().isin(["true", "1"]), "review", "routine")
by_date = {t: {d: g.set_index("unit_id") for d, g in sel[t].groupby("date")} for t in sel}


def board_for(date):
    out = []
    for uid in sorted(by_date["m2"][date].index):
        entry = {"unit_id": uid, "m1": None, "m2": None}
        for t in ("m1", "m2"):
            frame = by_date[t].get(date)
            if frame is not None and uid in frame.index:
                entry[t] = {"tier": frame.loc[uid, "tier"], "score": float(frame.loc[uid, "score"])}
        out.append(entry)
    return out


rows = []
for date in sorted(by_date["m2"]):
    board = board_for(date)
    for task in ("m1", "m2"):
        groups = cl.hotspot_clusters(units, board, task)
        flagged = sum(1 for b in board if b[task] and b[task]["tier"] == "review")
        rows.append({"date": date, "task": task.upper(), "units in review": flagged, "hotspot groups": len(groups),
                     "units inside a hotspot": sum(g["size"] for g in groups), "largest hotspot": max([g["size"] for g in groups], default=0)})
hot_df = pd.DataFrame(rows)
summary_hot = hot_df.groupby("task").agg(weeks=("date", "count"), mean_in_review=("units in review", "mean"),
                                        weeks_with_hotspot=("hotspot groups", lambda s: int((s > 0).sum())),
                                        mean_largest=("largest hotspot", "mean"),
                                        weeks_with_blob_10plus=("largest hotspot", lambda s: int((s >= 10).sum()))).round(2)
print("=" * 70); print(f"6. HOTSPOTS ({hot_world}, {hot_scenario} surveillance, {hot_df['date'].nunique()} weeks)"); print("=" * 70)
display(summary_hot)
hot_df.to_csv("review3_cluster_hotspots_by_week.csv", index=False)

for task, r in summary_hot.iterrows():
    print(f"{task}: on average {r['mean_in_review']:.1f} units are in Review per week; a hotspot of 10+ adjacent units "
          f"appears in {int(r['weeks_with_blob_10plus'])} of {int(r['weeks'])} weeks.")
print("Reading: when Review flags are rare (M1) a hotspot is a meaningful signal. When they are common (M2) adjacent flags chain "
      "into state-wide blobs, so hotspots are less specific for M2 and should be read together with the scores.")

# one worked example: the week with the most units in the Review tier for M2
peak = hot_df[hot_df["task"] == "M2"].sort_values("units in review", ascending=False).iloc[0]["date"]
example = cl.hotspot_clusters(units, board_for(peak), "m2")
name_of = {u["unit_id"]: u["name"].replace("_", " ") for u in units}
print(f"\nExample week {peak} (most M2 Review units): {len(example)} hotspot group(s)")
for g in example:
    print(f"  - {g['size']} adjacent units, {g['review_count']} in Review, top score {g['max_score']}: "
          + ", ".join(name_of[u] for u in g["units"]))

fig, ax = plt.subplots(figsize=(11, 3.8))
for task, colour in (("M1", "#FB8C00"), ("M2", "#1565C0")):
    sub = hot_df[hot_df["task"] == task]
    ax.plot(pd.to_datetime(sub["date"]), sub["largest hotspot"], lw=1.5, color=colour, label=f"{task}: largest hotspot (units)")
ax.set_ylabel("Adjacent Review-tier units"); ax.legend()
ax.set_title("Size of the largest group of adjacent flagged units, by week (test world)", fontsize=11, weight="bold")
plt.tight_layout(); plt.savefig("review3_cluster_hotspots.png", dpi=300); plt.show()
print("Saved review3_cluster_hotspots.png and review3_cluster_hotspots_by_week.csv")

In [ ]:
# ==============================================================================
# REVIEW III - PART 5: MASTER SUMMARY TABLE & ONE-CLICK ZIP DOWNLOAD
# Every "Key finding" below is computed from this run's results (nothing is typed in by hand).
# ==============================================================================
import glob, os, shutil

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

metric_lookup = {}                                    # model name -> its metrics row
for table, key in [(part1_df, "Model"), (part3_df, "Model Architecture")]:
    for _, r in table.iterrows():
        metric_lookup[r[key]] = r


def m(name, col):
    return metric_lookup[name][col]


def summary(name):
    return f"F1 {m(name, 'Test F1-Score'):.3f}, Recall {m(name, 'Test Recall'):.3f}, ROC-AUC {m(name, 'Test ROC-AUC'):.3f}"


n_pc_90 = int(np.searchsorted(cum_var, 90) + 1) if cum_var[-1] >= 90 else None
rows = [
    {"Category": "1. Dimensionality reduction", "Model/Method": f"PCA ({N_PC} components)",
     "Key Finding/Metric": f"5 PCs explain {cum_var[4]:.1f}% of variance"
                           + (f"; {n_pc_90} PCs reach 90%" if n_pc_90 else f"; {N_PC} PCs explain {cum_var[-1]:.1f}%")},
    {"Category": "2. Distance classifier", "Model/Method": "K-Nearest Neighbours (k=5)",
     "Key Finding/Metric": summary("K-Nearest Neighbours (k=5)")},
]
for nb_name in ["Gaussian Naive Bayes", "Bernoulli Naive Bayes", "Multinomial Naive Bayes"]:
    rows.append({"Category": "3. Naive Bayes", "Model/Method": nb_name, "Key Finding/Metric": summary(nb_name)})
for svm_name in ["SVM (Linear Kernel)", "SVM (RBF Kernel)", "SVM (Polynomial Kernel d=3)"]:
    rows.append({"Category": "4. Support Vector Machine", "Model/Method": svm_name, "Key Finding/Metric": summary(svm_name)})
rows += [
    {"Category": "5. Clustering", "Model/Method": f"K-Means (elbow choice K={optimal_k})",
     "Key Finding/Metric": f"Inertia {km_inertia:.1f}, silhouette {km_sil:.3f}, Davies-Bouldin {km_db:.3f}; "
                           f"outbreak rate by cluster {km_profile['Outbreak Positive Rate'].min():.1%} to "
                           f"{km_profile['Outbreak Positive Rate'].max():.1%}"},
    {"Category": "5. Clustering", "Model/Method": f"Hierarchical (Ward, K={optimal_k})",
     "Key Finding/Metric": f"Silhouette {hier_sil:.3f}, Davies-Bouldin {hier_db:.3f}; outbreak rate by cluster "
                           f"{hier_profile['Outbreak Positive Rate'].min():.1%} to {hier_profile['Outbreak Positive Rate'].max():.1%}"},
]
for tree_name in ["Decision Tree (Gini Impurity)", "Decision Tree (Information Gain / Entropy)", "Random Forest (Bagging)"]:
    rows.append({"Category": "6. Tree models", "Model/Method": tree_name, "Key Finding/Metric": summary(tree_name)})
for nn_name in ["Single-Layer Perceptron", "Artificial Neural Network (MLP 64-32)"]:
    rows.append({"Category": "7. Neural networks", "Model/Method": nn_name, "Key Finding/Metric": summary(nn_name)})

all_models = pd.concat([part1_df.rename(columns={"Model": "Model Architecture"}), part3_df], ignore_index=True)
best_row = all_models.loc[all_models["Test ROC-AUC"].idxmax()]
rows.append({"Category": "8. Best of Review III", "Model/Method": best_row["Model Architecture"],
             "Key Finding/Metric": f"Highest test ROC-AUC ({best_row['Test ROC-AUC']:.3f}), "
                                   f"F1 {best_row['Test F1-Score']:.3f}"})

# ---- clustering used inside the product (every number comes from the cells above) ----
cl_blind = cluster_report["blind_spots"]["clusters"]
cl_reg = cluster_report["regimes"]["clusters"]
rows += [
    {"Category": "9. Clustering (product)", "Model/Method": "Area archetypes (K-means)",
     "Key Finding/Metric": f"K={cluster_report['archetypes']['k']}, silhouette {cluster_report['archetypes']['quality']['silhouette']:.2f}; "
                           f"{len(units)} units grouped by size, density, coast, water mix and sanitation"},
    {"Category": "9. Clustering (product)", "Model/Method": "Weekly regimes (K-means)",
     "Key Finding/Metric": f"K={cluster_report['regimes']['k']}, silhouette {cluster_report['regimes']['silhouette']:.2f}, seed agreement ARI {ari_seed:.2f}; "
                           f"outbreak rate {cl_reg[0]['m2_active_rate']:.0%} (calmest) to {cl_reg[-1]['m2_active_rate']:.0%} (hottest)"},
    {"Category": "9. Clustering (product)", "Model/Method": "Surveillance blind spots (K-means, K=3)",
     "Key Finding/Metric": f"events never reported: {cl_blind[0]['unreported_share']:.0%} (well covered) to {cl_blind[-1]['unreported_share']:.0%} (blind spot)"},
    {"Category": "9. Clustering (product)", "Model/Method": "Seasonal patterns (K-means)",
     "Key Finding/Metric": f"K={cluster_report['seasonal']['k']}, silhouette {cluster_report['seasonal']['quality']['silhouette']:.2f}: "
                           + "; ".join(f"{c['name']} ({c['size']} units)" for c in cluster_report["seasonal"]["clusters"])},
    {"Category": "9. Clustering (product)", "Model/Method": "Outbreak types (K-means)",
     "Key Finding/Metric": (f"K={cluster_report['event_types']['k']}, silhouette {cluster_report['event_types']['silhouette']:.2f}, "
                            f"{cluster_report['event_types']['events']} test events") if cluster_report["event_types"].get("available") else "too few events to cluster"},
    {"Category": "9. Clustering (product)", "Model/Method": "Hotspots (DBSCAN on neighbour graph)",
     "Key Finding/Metric": "; ".join(f"{t}: hotspot in {int(r.weeks_with_hotspot)} of {int(r.weeks)} weeks, 10+ unit blob in {int(r.weeks_with_blob_10plus)}"
                                     for t, r in summary_hot.iterrows())},
]

review3_master = pd.DataFrame(rows)
print("=" * 75)
print("REVIEW III MASTER EVALUATION SUBMISSION TABLE")
print("=" * 75)
pd.set_option("display.max_colwidth", 200)
display(review3_master)
review3_master.to_csv("review3_master_summary.csv", index=False)

# ---- zip every Review III plot and table ----
OUT_ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
pack_dir_r3 = os.path.join(OUT_ROOT, "sanjeevani_review3_pack")
shutil.rmtree(pack_dir_r3, ignore_errors=True)
os.makedirs(pack_dir_r3)
for path in sorted(glob.glob("review3_*.png") + glob.glob("review3_*.csv")):
    shutil.copy(path, pack_dir_r3)
    print("  packed", path)

archive_r3 = shutil.make_archive(os.path.join(OUT_ROOT, "Sanjeevani_Review3_Complete_Pack"), "zip", pack_dir_r3)
print("\nCreated:", archive_r3)
if IN_COLAB:
    files.download(archive_r3)
    print("Download started - all Review III plots and CSVs are going to your computer.")